In [1]:
!pip install pypdf

In [2]:
#injestion and chunking

from pathlib import Path
import re
from pypdf import PdfReader
from transformers import AutoTokenizer

PDF_PATH = Path("Southwind Labs Company Documents.pdf")
EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"
CHUNK_SIZE = 450
CHUNK_OVERLAP = 80
STRIDE = CHUNK_SIZE - CHUNK_OVERLAP  # 370

assert PDF_PATH.exists(), f"Missing {PDF_PATH.resolve()}"

tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_NAME)
reader = PdfReader(str(PDF_PATH))

pages_text = []
for page_idx, page in enumerate(reader.pages, start=1):
    extracted = (page.extract_text() or "").strip()
    if extracted:
        pages_text.append((page_idx, extracted))

raw_text = "\n\n".join(f"[Page {p}]\n{txt}" for p, txt in pages_text)


def infer_roles(text: str) -> list[str]:
    lower = text.lower()
    if any(
        k in lower
        for k in [
            "admin-only",
            "strictly confidential",
            "board",
            "salary",
            "bank account",
            "wire",
        ]
    ):
        return ["admin"]
    if any(
        k in lower
        for k in [
            "manager",
            "confidential - managers",
            "compensation band",
            "performance review",
        ]
    ):
        return ["manager", "admin"]
    return ["employee", "manager", "admin"]


sections = []
current_title = "Page 1"
current_lines = []

for line in raw_text.splitlines():
    page_match = re.match(r"^\[Page (\d+)\]$", line.strip())
    heading_match = re.match(
        r"^(?:#{1,3}\s+|SECTION\s+\d+\b|[A-Z][A-Z\s/&\-]{5,}$)", line.strip()
    )
    if page_match or heading_match:
        if current_lines:
            sections.append((current_title, "\n".join(current_lines).strip()))
        current_title = line.strip()
        current_lines = [line]
    else:
        current_lines.append(line)

if current_lines:
    sections.append((current_title, "\n".join(current_lines).strip()))

chunks = []
for sec_idx, (sec_title, sec_text) in enumerate(sections):
    if not sec_text:
        continue
    token_ids = tokenizer.encode(sec_text, add_special_tokens=False)
    allowed_roles = infer_roles(sec_text)

    if len(token_ids) <= CHUNK_SIZE:
        chunk_text = tokenizer.decode(token_ids, skip_special_tokens=True)
        chunks.append(
            {
                "chunk_id": f"sec{sec_idx}_chk0",
                "section": sec_title,
                "allowed_roles": allowed_roles,
                "token_count": len(token_ids),
                "text": chunk_text,
            }
        )
    else:
        start = 0
        chk_num = 0
        while start < len(token_ids):
            window_ids = token_ids[start : start + CHUNK_SIZE]
            chunk_text = tokenizer.decode(window_ids, skip_special_tokens=True)
            chunks.append(
                {
                    "chunk_id": f"sec{sec_idx}_chk{chk_num}",
                    "section": sec_title,
                    "allowed_roles": allowed_roles,
                    "token_count": len(window_ids),
                    "text": chunk_text,
                }
            )
            if start + CHUNK_SIZE >= len(token_ids):
                break
            start += STRIDE
            chk_num += 1

print(f"Pages extracted: {len(pages_text)}")
print(f"Sections found: {len(sections)}")
print(f"Total chunks created: {len(chunks)}")
print(f"Max chunk tokens: {max((c['token_count'] for c in chunks), default=0)}")
for sample in chunks[:3]:
    print(
        f"- {sample['chunk_id']} | roles={sample['allowed_roles']} | "
        f"tokens={sample['token_count']} | section={sample['section'][:50]}"
    )

C:\Users\Lenovo\Desktop\hybrid-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pages extracted: 16
Sections found: 16
Total chunks created: 16
Max chunk tokens: 434
- sec0_chk0 | roles=['manager', 'admin'] | tokens=372 | section=[Page 1]
- sec1_chk0 | roles=['manager', 'admin'] | tokens=336 | section=[Page 2]
- sec2_chk0 | roles=['admin'] | tokens=428 | section=[Page 3]


In [4]:
#EMBEDDINGS
import json
import pickle
from pathlib import Path
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

CHUNKS_JSON_PATH = Path("chunks.json")
QDRANT_PATH = Path("qdrant_data")
BM25_PKL_PATH = Path("bm25_index.pkl")
COLLECTION_NAME = "southwind_docs"

assert len(chunks) > 0, "No chunks found. Run the chunking cell first."

# 1. Persist chunks to JSON for downstream retrieval / evaluation / API stages
CHUNKS_JSON_PATH.write_text(json.dumps(chunks, indent=2, ensure_ascii=False), encoding="utf-8")

# 2. Compute dense embeddings locally on CPU
embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cpu")
texts = [c["text"] for c in chunks]
embeddings = embed_model.encode(texts, batch_size=8, normalize_embeddings=True, show_progress_bar=False)
vector_dim = embeddings.shape[1]

# 3. Store vectors + RBAC payload in local Qdrant
qdrant = QdrantClient(path=str(QDRANT_PATH))
if qdrant.collection_exists(COLLECTION_NAME):
    qdrant.delete_collection(COLLECTION_NAME)

qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=vector_dim, distance=Distance.COSINE),
)

points = [
    PointStruct(
        id=idx,
        vector=vec.tolist(),
        payload={
            "chunk_id": chunk["chunk_id"],
            "section": chunk["section"],
            "allowed_roles": chunk["allowed_roles"],
            "token_count": chunk["token_count"],
            "text": chunk["text"],
        },
    )
    for idx, (chunk, vec) in enumerate(zip(chunks, embeddings))
]
qdrant.upsert(collection_name=COLLECTION_NAME, points=points)

# 4. Build BM25 index using the same tokenizer subwords/terms for consistent lexical matching
tokenized_corpus = [tokenizer.tokenize(c["text"].lower()) for c in chunks]
bm25 = BM25Okapi(tokenized_corpus)
with open(BM25_PKL_PATH, "wb") as f:
    pickle.dump({"bm25": bm25, "chunks": chunks, "tokenized_corpus": tokenized_corpus}, f)

count_info = qdrant.count(collection_name=COLLECTION_NAME)
print(f"Saved {len(chunks)} chunks to {CHUNKS_JSON_PATH}")
print(f"Qdrant collection '{COLLECTION_NAME}' vector dim={vector_dim}, points={count_info.count}")
print(f"Saved BM25 index to {BM25_PKL_PATH} (docs={len(tokenized_corpus)})")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 918.72it/s]


Saved 16 chunks to chunks.json
Qdrant collection 'southwind_docs' vector dim=384, points=16
Saved BM25 index to bm25_index.pkl (docs=16)


In [6]:
#INSPECTION and CHECKING
import json
import pickle
from qdrant_client import QdrantClient
from qdrant_client.models import FieldCondition, Filter, MatchAny
from sentence_transformers import SentenceTransformer

# 1. Inspect all 16 chunks (roles, token count, preview)
loaded_chunks = json.loads(Path("chunks.json").read_text(encoding="utf-8"))
print("=== CHUNK OVERVIEW ===")
for c in loaded_chunks:
    single_line = " ".join(c["text"].split())
    print(f"{c['chunk_id']:12s} | roles={str(c['allowed_roles']):35s} | tok={c['token_count']:3d} | {single_line[:90]}")

# 2. Smoke-test dense Qdrant search with RBAC filtering
embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cpu")
qdrant = QdrantClient(path="qdrant_data")

test_query = "salary bank account wire transfer policy"
q_vec = embed_model.encode(test_query, normalize_embeddings=True).tolist()

for role in ["employee", "manager", "admin"]:
    hits = qdrant.query_points(
        collection_name="southwind_docs",
        query=q_vec,
        query_filter=Filter(
            must=[FieldCondition(key="allowed_roles", match=MatchAny(any=[role]))]
        ),
        limit=2,
    ).points
    top_summary = [(h.payload["chunk_id"], round(h.score, 3), h.payload["allowed_roles"]) for h in hits]
    print(f"\n[Qdrant role='{role}'] top hits -> {top_summary}")

# 3. Smoke-test BM25 lexical scores
with open("bm25_index.pkl", "rb") as f:
    bm25_bundle = pickle.load(f)

bm25 = bm25_bundle["bm25"]
q_tokens = tokenizer.tokenize(test_query.lower())
scores = bm25.get_scores(q_tokens)
ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:3]
print("\n[BM25 top chunks] ->", [(loaded_chunks[i]["chunk_id"], round(float(s), 3), loaded_chunks[i]["allowed_roles"]) for i, s in ranked])

=== CHUNK OVERVIEW ===
sec0_chk0    | roles=['manager', 'admin']                | tok=372 | [ page 1 ] southwind labs company documents nov 25, 2026 · @ subb fictional knowledge base
sec1_chk0    | roles=['manager', 'admin']                | tok=336 | [ page 2 ] unused casual and sick leave lapse on 31 march. earned leave can be encashed on
sec2_chk0    | roles=['admin']                           | tok=428 | [ page 3 ] or 2 triggers a 90 - day improvement plan agreed between the employee, manager 
sec3_chk0    | roles=['admin']                           | tok=395 | [ page 4 ] document 2 : information security policy access header : department security · 
sec4_chk0    | roles=['manager', 'admin']                | tok=330 | [ page 5 ] 5. device and network security all laptops use full - disk encryption and autom
sec5_chk0    | roles=['employee', 'manager', 'admin']    | tok=396 | [ page 6 ] 8. vendor and third - party risk new vendors that process company or customer d
sec6_chk0    | ro

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 941.98it/s]


RuntimeError: Storage folder qdrant_data is already accessed by another instance of Qdrant client. If you require concurrent access, use Qdrant server instead.

In [7]:
#smoked and reusing qudrant (2 instance running for one client error)
import json
import pickle
from qdrant_client import QdrantClient
from qdrant_client.models import FieldCondition, Filter, MatchAny
from sentence_transformers import SentenceTransformer

# Reuse existing Qdrant client in the kernel if already created
if "qdrant" in globals() and qdrant is not None:
    client = qdrant
else:
    client = QdrantClient(path="qdrant_data")
    

if "embed_model" not in globals() or embed_model is None:
    embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cpu")

test_query = "salary bank account wire transfer policy"
q_vec = embed_model.encode(test_query, normalize_embeddings=True).tolist()

for role in ["employee", "manager", "admin"]:
    hits = client.query_points(
        collection_name="southwind_docs",
        query=q_vec,
        query_filter=Filter(
            must=[FieldCondition(key="allowed_roles", match=MatchAny(any=[role]))]
        ),
        limit=2,
    ).points
    top_summary = [
        (h.payload["chunk_id"], round(h.score, 3), h.payload["allowed_roles"])
        for h in hits
    ]
    print(f"[Qdrant role='{role}'] top hits -> {top_summary}")

with open("bm25_index.pkl", "rb") as f:
    bm25_bundle = pickle.load(f)

bm25 = bm25_bundle["bm25"]
q_tokens = tokenizer.tokenize(test_query.lower())
scores = bm25.get_scores(q_tokens)
ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:3]
print(
    "\n[BM25 top chunks] ->",
    [
        (loaded_chunks[i]["chunk_id"], round(float(s), 3), loaded_chunks[i]["allowed_roles"])
        for i, s in ranked
    ],
)

[Qdrant role='employee'] top hits -> [('sec5_chk0', 0.57, ['employee', 'manager', 'admin']), ('sec6_chk0', 0.548, ['employee', 'manager', 'admin'])]
[Qdrant role='manager'] top hits -> [('sec1_chk0', 0.591, ['manager', 'admin']), ('sec0_chk0', 0.582, ['manager', 'admin'])]
[Qdrant role='admin'] top hits -> [('sec12_chk0', 0.685, ['admin']), ('sec2_chk0', 0.653, ['admin'])]

[BM25 top chunks] -> [('sec12_chk0', 6.036, ['admin']), ('sec3_chk0', 2.931, ['admin']), ('sec2_chk0', 2.161, ['admin'])]


In [8]:
from pathlib import Path
from pypdf import PdfReader

reader = PdfReader("Southwind Labs Company Documents.pdf")
for p_idx, page in enumerate(reader.pages, start=1):
    for line in (page.extract_text() or "").splitlines():
        s = line.strip()
        if any(k in s.lower() for k in ["document ", "access", "role", "confidential"]):
            print(f"P{p_idx:02d}: {s[:120]}")

P01: software company in Bengaluru with about 300 employees. Each document carries an
P01: access header (allowed roles and sensitivity) that becomes chunk metadata at ingestion.
P01: Document 1: HR Policy Handbook
P01: Access header: department hr · allowed_roles employee, manager, admin · sensitivity
P01: internal. Section 7 (Compensation Bands) is a separate chunk group: allowed_roles
P01: Document ID: HR-001 · Owner: People Operations · Version 3.2 · Effective 1 April 2026
P03: The bands below are annual fixed pay in Indian rupees and are confidential. They must not
P03: be shared with employees outside the manager and admin roles, and must not be quoted
P03: revoke personal access to company systems on their last day.
P04: Document 2: Information Security Policy
P04: Access header: department security · allowed_roles employee, manager, admin ·
P04: allowed_roles admin · sensitivity restricted · contains_planted_secret true.
P04: Document ID: SEC-001 · Owner: Security Team · Versio

In [13]:
import json
import pickle
import re
from pathlib import Path
from pypdf import PdfReader
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer

PDF_PATH = Path("Southwind Labs Company Documents.pdf")
EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"
CHUNKS_JSON_PATH = Path("chunks.json")
QDRANT_PATH = Path("qdrant_data")
BM25_PKL_PATH = Path("bm25_index.pkl")
COLLECTION_NAME = "southwind_docs"
CHUNK_SIZE = 450
CHUNK_OVERLAP = 80
STRIDE = CHUNK_SIZE - CHUNK_OVERLAP

tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_NAME)
reader = PdfReader(str(PDF_PATH))

# Document-level baseline roles by Document ID / header
DOC_BASE_ROLES = {
    "HR-001": ["employee", "manager", "admin"],
    "SEC-001": ["employee", "manager", "admin"],
    "ENG-001": ["employee", "manager", "admin"],
    "FIN-001": ["manager", "admin"],
    "LDR-001": ["admin"],
}


def resolve_roles(doc_id: str | None, text: str) -> list[str]:
    base = list(DOC_BASE_ROLES.get(doc_id or "", ["employee", "manager", "admin"]))
    low = text.lower()

    # Section-level overrides inside documents
    if any(
        marker in low
        for marker in [
            "9. confidential incident review",
            "8. production access (admins only)",
            "6. executive compensation and bank details",
            "5. confidential: acquisition plan (admins only)",
            "restricted to the admin role",
        ]
    ):
        return ["admin"]

    if "7. compensation bands" in low or "compensation bands" in low and "annual fixed pay" in low:
        return ["manager", "admin"]

    if "roadmap is shared with all employees" in low or "public product roadmap" in low:
        return ["employee", "manager", "admin"]

    return base


current_doc_id: str | None = None
chunks = []

for page_idx, page in enumerate(reader.pages, start=1):
    raw_page = (page.extract_text() or "").strip()
    if not raw_page:
        continue

    doc_match = re.search(r"\b(HR-001|SEC-001|ENG-001|FIN-001|LDR-001)\b", raw_page)
    if doc_match:
        current_doc_id = doc_match.group(1)

    section_text = f"[Page {page_idx}]\n{raw_page}"
    allowed_roles = resolve_roles(current_doc_id, section_text)
    token_ids = tokenizer.encode(section_text, add_special_tokens=False)

    if len(token_ids) <= CHUNK_SIZE:
        windows = [token_ids]
    else:
        windows = []
        start = 0
        while start < len(token_ids):
            windows.append(token_ids[start : start + CHUNK_SIZE])
            if start + CHUNK_SIZE >= len(token_ids):
                break
            start += STRIDE

    for chk_num, win_ids in enumerate(windows):
        chunks.append(
            {
                "chunk_id": f"sec{page_idx - 1}_chk{chk_num}",
                "doc_id": current_doc_id,
                "section": f"[Page {page_idx}]",
                "allowed_roles": allowed_roles,
                "token_count": len(win_ids),
                "text": tokenizer.decode(win_ids, skip_special_tokens=True),
            }
        )

CHUNKS_JSON_PATH.write_text(json.dumps(chunks, indent=2, ensure_ascii=False), encoding="utf-8")

embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cpu")
embeddings = embed_model.encode(
    [c["text"] for c in chunks],
    batch_size=8,
    normalize_embeddings=True,
    show_progress_bar=False,
)

qdrant = QdrantClient(path=str(QDRANT_PATH))
if qdrant.collection_exists(COLLECTION_NAME):
    qdrant.delete_collection(COLLECTION_NAME)
qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=embeddings.shape[1], distance=Distance.COSINE),
)
qdrant.upsert(
    collection_name=COLLECTION_NAME,
    points=[
        PointStruct(
            id=i,
            vector=vec.tolist(),
            payload={
                "chunk_id": c["chunk_id"],
                "doc_id": c["doc_id"],
                "section": c["section"],
                "allowed_roles": c["allowed_roles"],
                "token_count": c["token_count"],
                "text": c["text"],
            },
        )
        for i, (c, vec) in enumerate(zip(chunks, embeddings))
    ],
)

tokenized_corpus = [tokenizer.tokenize(c["text"].lower()) for c in chunks]
bm25 = BM25Okapi(tokenized_corpus)
with open(BM25_PKL_PATH, "wb") as f:
    pickle.dump({"bm25": bm25, "chunks": chunks, "tokenized_corpus": tokenized_corpus}, f)

for c in chunks:
    print(f"{c['chunk_id']:12s} | doc={str(c['doc_id']):7s} | roles={str(c['allowed_roles']):35s} | tok={c['token_count']:3d}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 876.32it/s]


RuntimeError: Storage folder qdrant_data is already accessed by another instance of Qdrant client. If you require concurrent access, use Qdrant server instead.

In [10]:
import os
from pathlib import Path

qdrant_dir = Path("qdrant_data")
print("Exists:", qdrant_dir.exists())
if qdrant_dir.exists():
    for p in qdrant_dir.rglob("*"):
        if p.is_file():
            print(" ", p.relative_to(qdrant_dir), "size=", p.stat().st_size)

Exists: True
  .lock size= 13
  meta.json size= 587
  collection\southwind_docs\storage.sqlite size= 110592


In [12]:
from pathlib import Path
import shutil
from qdrant_client import QdrantClient

# If an earlier variable named `qdrant` exists in this kernel, close it first
if "qdrant" in globals() and globals()["qdrant"] is not None:
    try:
        globals()["qdrant"].close()
    except Exception:
        pass

# Test opening QdrantClient cleanly
client = QdrantClient(path="qdrant_data")
print("Opened qdrant_data successfully! Collections:", client.get_collections())

Opened qdrant_data successfully! Collections: collections=[CollectionDescription(name='southwind_docs')]


In [14]:
import json
import pickle
import re
from pathlib import Path
from pypdf import PdfReader
from qdrant_client.models import Distance, PointStruct, VectorParams
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer

PDF_PATH = Path("Southwind Labs Company Documents.pdf")
EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"
CHUNKS_JSON_PATH = Path("chunks.json")
BM25_PKL_PATH = Path("bm25_index.pkl")
COLLECTION_NAME = "southwind_docs"
CHUNK_SIZE = 450
CHUNK_OVERLAP = 80
STRIDE = CHUNK_SIZE - CHUNK_OVERLAP

tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_NAME)
reader = PdfReader(str(PDF_PATH))

DOC_BASE_ROLES = {
    "HR-001": ["employee", "manager", "admin"],
    "SEC-001": ["employee", "manager", "admin"],
    "ENG-001": ["employee", "manager", "admin"],
    "FIN-001": ["manager", "admin"],
    "LDR-001": ["admin"],
}


def resolve_roles(doc_id: str | None, text: str) -> list[str]:
    base = list(DOC_BASE_ROLES.get(doc_id or "", ["employee", "manager", "admin"]))
    low = text.lower()

    if any(
        marker in low
        for marker in [
            "9. confidential incident review",
            "8. production access (admins only)",
            "6. executive compensation and bank details",
            "5. confidential: acquisition plan (admins only)",
            "restricted to the admin role",
        ]
    ):
        return ["admin"]

    if "7. compensation bands" in low or ("compensation bands" in low and "annual fixed pay" in low):
        return ["manager", "admin"]

    if "roadmap is shared with all employees" in low or "public product roadmap" in low:
        return ["employee", "manager", "admin"]

    return base


current_doc_id: str | None = None
chunks = []

for page_idx, page in enumerate(reader.pages, start=1):
    raw_page = (page.extract_text() or "").strip()
    if not raw_page:
        continue

    doc_match = re.search(r"\b(HR-001|SEC-001|ENG-001|FIN-001|LDR-001)\b", raw_page)
    if doc_match:
        current_doc_id = doc_match.group(1)

    section_text = f"[Page {page_idx}]\n{raw_page}"
    allowed_roles = resolve_roles(current_doc_id, section_text)
    token_ids = tokenizer.encode(section_text, add_special_tokens=False)

    if len(token_ids) <= CHUNK_SIZE:
        windows = [token_ids]
    else:
        windows = []
        start = 0
        while start < len(token_ids):
            windows.append(token_ids[start : start + CHUNK_SIZE])
            if start + CHUNK_SIZE >= len(token_ids):
                break
            start += STRIDE

    for chk_num, win_ids in enumerate(windows):
        chunks.append(
            {
                "chunk_id": f"sec{page_idx - 1}_chk{chk_num}",
                "doc_id": current_doc_id,
                "section": f"[Page {page_idx}]",
                "allowed_roles": allowed_roles,
                "token_count": len(win_ids),
                "text": tokenizer.decode(win_ids, skip_special_tokens=True),
            }
        )

CHUNKS_JSON_PATH.write_text(json.dumps(chunks, indent=2, ensure_ascii=False), encoding="utf-8")

embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cpu")
embeddings = embed_model.encode(
    [c["text"] for c in chunks],
    batch_size=8,
    normalize_embeddings=True,
    show_progress_bar=False,
)

if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)
client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=embeddings.shape[1], distance=Distance.COSINE),
)
client.upsert(
    collection_name=COLLECTION_NAME,
    points=[
        PointStruct(
            id=i,
            vector=vec.tolist(),
            payload={
                "chunk_id": c["chunk_id"],
                "doc_id": c["doc_id"],
                "section": c["section"],
                "allowed_roles": c["allowed_roles"],
                "token_count": c["token_count"],
                "text": c["text"],
            },
        )
        for i, (c, vec) in enumerate(zip(chunks, embeddings))
    ],
)

tokenized_corpus = [tokenizer.tokenize(c["text"].lower()) for c in chunks]
bm25 = BM25Okapi(tokenized_corpus)
with open(BM25_PKL_PATH, "wb") as f:
    pickle.dump({"bm25": bm25, "chunks": chunks, "tokenized_corpus": tokenized_corpus}, f)

for c in chunks:
    print(f"{c['chunk_id']:12s} | doc={str(c['doc_id']):7s} | roles={str(c['allowed_roles']):35s} | tok={c['token_count']:3d}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1233.90it/s]


sec0_chk0    | doc=HR-001  | roles=['employee', 'manager', 'admin']    | tok=372
sec1_chk0    | doc=HR-001  | roles=['employee', 'manager', 'admin']    | tok=336
sec2_chk0    | doc=HR-001  | roles=['manager', 'admin']                | tok=428
sec3_chk0    | doc=SEC-001 | roles=['employee', 'manager', 'admin']    | tok=395
sec4_chk0    | doc=SEC-001 | roles=['employee', 'manager', 'admin']    | tok=330
sec5_chk0    | doc=SEC-001 | roles=['admin']                           | tok=396
sec6_chk0    | doc=SEC-001 | roles=['employee', 'manager', 'admin']    | tok=331
sec7_chk0    | doc=ENG-001 | roles=['employee', 'manager', 'admin']    | tok=342
sec8_chk0    | doc=ENG-001 | roles=['employee', 'manager', 'admin']    | tok=428
sec9_chk0    | doc=ENG-001 | roles=['employee', 'manager', 'admin']    | tok=338
sec10_chk0   | doc=FIN-001 | roles=['admin']                           | tok=434
sec11_chk0   | doc=FIN-001 | roles=['manager', 'admin']                | tok=315
sec12_chk0   | doc=FIN-001 |